<a href="https://colab.research.google.com/github/AsimaZaheer/Stress-Detection/blob/main/CNN%20integrated%20murtaza%20V1%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# WESAD STRESS DETECTION
# 32 Hz + 3-Layer 1D CNN + HRV
# LOSO (Leave-One-Subject-Out) Validation
# ============================================================

# IMPORTANT:
# Run this entire cell as ONE Python cell.
# Do NOT copy the ``` lines into the notebook.
# ============================================================

import os
import pickle
import zipfile
import subprocess
import sys

import numpy as np
from scipy import signal
from scipy.signal import find_peaks

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# 1. CONFIGURATION
# ============================================================

SUBJECTS = [
    "S2", "S3", "S4", "S5", "S6", "S7",
    "S8", "S9", "S10", "S11", "S13", "S14",
    "S15", "S16", "S17"
]

TARGET_SAMPLING_RATE = 32
LABEL_NATIVE_RATE = 700

WINDOW_SIZE = 60       # seconds
STEP_SIZE = 10         # seconds

BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-3

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Google Drive WESAD ZIP
WESAD_ZIP_URL = (
    "https://drive.google.com/file/d/"
    "1MbfU2z4OnyevB0oX_yvHVue7Wv24KP7l"
    "/view?usp=sharing"
)

ZIP_PATH = "/content/WESAD.zip"
EXTRACT_PATH = "/content"


# ============================================================
# 2. DOWNLOAD / EXTRACT DATASET
# ============================================================

def ensure_gdown():
    """Install gdown if it is not already installed."""
    try:
        import gdown
        return gdown
    except ImportError:
        print("Installing gdown...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "gdown"]
        )
        import gdown
        return gdown


def find_subject_file(dataset_root, subject):
    """Recursively find Sx.pkl inside the extracted WESAD folder."""
    target_name = f"{subject}.pkl"

    # First check common structure: root/S2/S2.pkl
    direct_path = os.path.join(
        dataset_root, subject, target_name
    )

    if os.path.isfile(direct_path):
        return direct_path

    # Then search recursively
    for root, _, files in os.walk(dataset_root):
        if target_name in files:
            return os.path.join(root, target_name)

    return None


def download_wesad_dataset():
    """Download and extract WESAD, then return dataset root."""

    print("=" * 70)
    print("WESAD DATASET SETUP")
    print("=" * 70)

    gdown = ensure_gdown()

    # --------------------------------------------------------
    # Download
    # --------------------------------------------------------
    if not os.path.isfile(ZIP_PATH):
        print("\nDownloading WESAD ZIP...")
        gdown.download(
            WESAD_ZIP_URL,
            ZIP_PATH,
            quiet=False,
            fuzzy=True
        )
    else:
        print("\nWESAD ZIP already exists:")
        print(ZIP_PATH)

    if not os.path.isfile(ZIP_PATH):
        raise FileNotFoundError(
            f"Download failed. ZIP was not created:\n{ZIP_PATH}"
        )

    # --------------------------------------------------------
    # Validate ZIP
    # --------------------------------------------------------
    if not zipfile.is_zipfile(ZIP_PATH):
        raise RuntimeError(
            "Downloaded file is not a valid ZIP file. "
            "Check your Google Drive link and file permissions."
        )

    # --------------------------------------------------------
    # Extract only if subject files are not already present
    # --------------------------------------------------------
    existing_s2 = find_subject_file(EXTRACT_PATH, "S2")

    if existing_s2 is None:
        print("\nExtracting WESAD ZIP...")
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(EXTRACT_PATH)
        print("Extraction completed.")
    else:
        print("\nWESAD appears to be already extracted.")

    # --------------------------------------------------------
    # Find S2.pkl
    # --------------------------------------------------------
    s2_file = find_subject_file(EXTRACT_PATH, "S2")

    if s2_file is None:
        raise FileNotFoundError(
            "\nS2.pkl was not found after extraction.\n"
            "Expected something like:\n"
            "/content/WESAD/S2/S2.pkl\n"
            "or:\n"
            "/content/S2/S2.pkl\n"
        )

    dataset_root = os.path.dirname(os.path.dirname(s2_file))

    print("\nS2.pkl found:")
    print(s2_file)

    print("\nDataset root:")
    print(dataset_root)

    return dataset_root


# ============================================================
# 3. SIGNAL RESAMPLING
# ============================================================

def resample_signal(x, original_fs, target_fs):
    """Resample a 1-D signal using scipy.signal.resample."""

    x = np.asarray(x, dtype=np.float32).squeeze()

    if x.ndim != 1:
        raise ValueError(
            f"Expected 1-D signal, got shape {x.shape}"
        )

    if len(x) == 0:
        return x

    if original_fs == target_fs:
        return x.astype(np.float32)

    new_length = int(
        round(len(x) * target_fs / original_fs)
    )

    new_length = max(new_length, 1)

    return signal.resample(
        x, new_length
    ).astype(np.float32)


# ============================================================
# 4. ALIGN 700 Hz LABELS TO 32 Hz
# ============================================================

def align_labels_by_time(
    labels,
    label_fs,
    target_fs,
    target_length
):
    """
    Convert native 700 Hz labels to target sampling rate
    by selecting the label corresponding to each target time.
    """

    labels = np.asarray(labels).squeeze()

    if labels.ndim != 1:
        raise ValueError(
            f"Labels must be 1-D, got {labels.shape}"
        )

    if len(labels) == 0:
        raise ValueError("Labels array is empty.")

    target_times = (
        np.arange(target_length, dtype=np.float64)
        / float(target_fs)
    )

    indices = np.floor(
        target_times * label_fs
    ).astype(np.int64)

    indices = np.clip(
        indices,
        0,
        len(labels) - 1
    )

    return labels[indices]


# ============================================================
# 5. HRV FEATURES
# ============================================================

def extract_hrv_features(bvp_window, fs):
    """
    Extract:
    1. Mean RR
    2. SDNN
    3. RMSSD
    """

    try:
        bvp_window = np.asarray(
            bvp_window,
            dtype=np.float32
        ).squeeze()

        if len(bvp_window) < 3:
            return np.zeros(3, dtype=np.float32)

        # Remove NaN / inf safely
        if not np.all(np.isfinite(bvp_window)):
            finite_mask = np.isfinite(bvp_window)

            if finite_mask.sum() < 3:
                return np.zeros(3, dtype=np.float32)

            bvp_window = np.interp(
                np.arange(len(bvp_window)),
                np.flatnonzero(finite_mask),
                bvp_window[finite_mask]
            )

        # Basic peak detection.
        # At 32 Hz, 0.4 sec corresponds to ~13 samples.
        min_distance = max(
            1,
            int(0.4 * fs)
        )

        # Use prominence relative to signal variation.
        signal_std = float(np.std(bvp_window))

        if signal_std > 0:
            prominence = 0.10 * signal_std
            peaks, _ = find_peaks(
                bvp_window,
                distance=min_distance,
                prominence=prominence
            )
        else:
            peaks, _ = find_peaks(
                bvp_window,
                distance=min_distance
            )

        if len(peaks) < 3:
            return np.zeros(3, dtype=np.float32)

        rr = np.diff(peaks).astype(np.float64) / float(fs)

        # Physiological sanity range:
        # 0.3 sec -> 200 BPM
        # 2.0 sec -> 30 BPM
        rr = rr[
            (rr >= 0.3) &
            (rr <= 2.0)
        ]

        if len(rr) < 2:
            return np.zeros(3, dtype=np.float32)

        mean_rr = np.mean(rr)
        sdnn = np.std(rr)

        diff_rr = np.diff(rr)

        if len(diff_rr) > 0:
            rmssd = np.sqrt(
                np.mean(diff_rr ** 2)
            )
        else:
            rmssd = 0.0

        features = np.array(
            [mean_rr, sdnn, rmssd],
            dtype=np.float32
        )

        if not np.all(np.isfinite(features)):
            return np.zeros(3, dtype=np.float32)

        return features

    except Exception:
        return np.zeros(3, dtype=np.float32)


# ============================================================
# 6. CREATE 60-SECOND WINDOWS
# ============================================================

def create_windows_with_features(
    eda,
    bvp,
    labels,
    fs,
    window_size_s=60,
    step_size_s=10
):
    """Create sequence, HRV and label arrays."""

    eda = np.asarray(eda, dtype=np.float32).squeeze()
    bvp = np.asarray(bvp, dtype=np.float32).squeeze()
    labels = np.asarray(labels).squeeze()

    common_length = min(
        len(eda),
        len(bvp),
        len(labels)
    )

    eda = eda[:common_length]
    bvp = bvp[:common_length]
    labels = labels[:common_length]

    window_samples = int(
        window_size_s * fs
    )

    step_samples = int(
        step_size_s * fs
    )

    if common_length < window_samples:
        return (
            np.empty(
                (0, window_samples, 2),
                dtype=np.float32
            ),
            np.empty(
                (0, 3),
                dtype=np.float32
            ),
            np.empty(
                (0,),
                dtype=np.int64
            )
        )

    sequences = []
    hrv_features = []
    window_labels = []

    for start in range(
        0,
        common_length - window_samples + 1,
        step_samples
    ):
        end = start + window_samples

        eda_win = eda[start:end]
        bvp_win = bvp[start:end]
        label_win = labels[start:end]

        # Only WESAD target classes:
        # 1 = Baseline
        # 2 = Stress
        # 3 = Amusement
        valid_labels = label_win[
            np.isin(label_win, [1, 2, 3])
        ]

        if len(valid_labels) == 0:
            continue

        values, counts = np.unique(
            valid_labels,
            return_counts=True
        )

        majority_label = int(
            values[np.argmax(counts)]
        )

        # EDA + BVP
        sequence = np.column_stack(
            [eda_win, bvp_win]
        ).astype(np.float32)

        # HRV
        hrv = extract_hrv_features(
            bvp_win,
            fs
        )

        # WESAD:
        # 1 -> 0 Baseline
        # 2 -> 1 Stress
        # 3 -> 2 Amusement
        class_label = majority_label - 1

        sequences.append(sequence)
        hrv_features.append(hrv)
        window_labels.append(class_label)

    if len(sequences) == 0:
        return (
            np.empty(
                (0, window_samples, 2),
                dtype=np.float32
            ),
            np.empty(
                (0, 3),
                dtype=np.float32
            ),
            np.empty(
                (0,),
                dtype=np.int64
            )
        )

    return (
        np.stack(sequences).astype(np.float32),
        np.stack(hrv_features).astype(np.float32),
        np.asarray(window_labels, dtype=np.int64)
    )


# ============================================================
# 7. CNN MODEL
# ============================================================

class StressClassifier(nn.Module):
    """
    Architecture:

    Input
      |
    Conv1D: 2 -> 32, kernel=7
      |
    ReLU
      |
    MaxPool1D(2)
      |
    Conv1D: 32 -> 64, kernel=5
      |
    ReLU
      |
    MaxPool1D(2)
      |
    Conv1D: 64 -> 128, kernel=3
      |
    ReLU
      |
    Global Average Pooling
      |
    Concatenate 3 HRV features
      |
    Dense 64
      |
    ReLU
      |
    Dropout 0.3
      |
    Output 3 classes
    """

    def __init__(
        self,
        seq_channels=2,
        hrv_dim=3,
        n_classes=3
    ):
        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv1d(
                in_channels=seq_channels,
                out_channels=32,
                kernel_size=7,
                padding=3
            ),

            nn.ReLU(),

            nn.MaxPool1d(
                kernel_size=2
            ),

            nn.Conv1d(
                in_channels=32,
                out_channels=64,
                kernel_size=5,
                padding=2
            ),

            nn.ReLU(),

            nn.MaxPool1d(
                kernel_size=2
            ),

            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                128 + hrv_dim,
                64
            ),

            nn.ReLU(),

            nn.Dropout(
                0.3
            ),

            nn.Linear(
                64,
                n_classes
            )
        )

    def forward(
        self,
        x_seq,
        x_hrv
    ):
        # x_seq:
        # [batch, time, channels]
        #
        # Conv1D requires:
        # [batch, channels, time]

        x = x_seq.permute(
            0, 2, 1
        )

        x = self.cnn(x)

        # [batch, 128, 1]
        # -> [batch, 128]

        x = x.squeeze(-1)

        # Add HRV:
        # [batch, 128] + [batch, 3]
        # -> [batch, 131]

        x = torch.cat(
            [x, x_hrv],
            dim=1
        )

        return self.classifier(x)


# ============================================================
# 8. WESAD PIPELINE
# ============================================================

class WESADPipeline:

    def __init__(self, dataset_path):
        self.dataset_path = dataset_path
        self.fs = TARGET_SAMPLING_RATE

    def load_subject_raw(self, subject):
        """Load wrist EDA, wrist BVP and labels."""

        file_path = find_subject_file(
            self.dataset_path,
            subject
        )

        if file_path is None:
            raise FileNotFoundError(
                f"\n{subject}.pkl not found under:\n"
                f"{self.dataset_path}"
            )

        print(f"\nLoading {subject}")
        print(f"File: {file_path}")

        with open(
            file_path,
            "rb"
        ) as f:
            data = pickle.load(
                f,
                encoding="latin1"
            )

        # ----------------------------------------------------
        # Check WESAD structure
        # ----------------------------------------------------
        if "signal" not in data:
            raise KeyError(
                f"{subject}: 'signal' key not found."
            )

        if "wrist" not in data["signal"]:
            raise KeyError(
                f"{subject}: wrist signal not found."
            )

        wrist = data["signal"]["wrist"]

        if "EDA" not in wrist:
            raise KeyError(
                f"{subject}: wrist EDA not found."
            )

        if "BVP" not in wrist:
            raise KeyError(
                f"{subject}: wrist BVP not found."
            )

        if "label" not in data:
            raise KeyError(
                f"{subject}: label not found."
            )

        eda_raw = np.asarray(
            wrist["EDA"],
            dtype=np.float32
        ).squeeze()

        bvp_raw = np.asarray(
            wrist["BVP"],
            dtype=np.float32
        ).squeeze()

        labels = np.asarray(
            data["label"]
        ).squeeze()

        # ----------------------------------------------------
        # Native WESAD rates
        # EDA = 4 Hz
        # BVP = 64 Hz
        # Labels = 700 Hz
        # ----------------------------------------------------

        eda = resample_signal(
            eda_raw,
            4,
            self.fs
        )

        bvp = resample_signal(
            bvp_raw,
            64,
            self.fs
        )

        # ----------------------------------------------------
        # Common signal length
        # ----------------------------------------------------

        common_length = min(
            len(eda),
            len(bvp)
        )

        eda = eda[:common_length]
        bvp = bvp[:common_length]

        # ----------------------------------------------------
        # Align labels to 32 Hz
        # ----------------------------------------------------

        labels = align_labels_by_time(
            labels,
            LABEL_NATIVE_RATE,
            self.fs,
            common_length
        )

        return eda, bvp, labels

    def build_all_subject_windows(self):
        """Build windows for all WESAD subjects."""

        all_sequences = []
        all_hrv = []
        all_labels = []
        all_groups = []

        for subject in SUBJECTS:

            print("\n" + "=" * 70)
            print(f"PROCESSING {subject}")
            print("=" * 70)

            eda, bvp, labels = (
                self.load_subject_raw(subject)
            )

            seq, hrv, y = (
                create_windows_with_features(
                    eda,
                    bvp,
                    labels,
                    self.fs,
                    WINDOW_SIZE,
                    STEP_SIZE
                )
            )

            if len(y) == 0:
                print(
                    f"WARNING: No valid windows found for {subject}."
                )
                continue

            all_sequences.append(seq)
            all_hrv.append(hrv)
            all_labels.append(y)

            all_groups.extend(
                [subject] * len(y)
            )

            print(
                f"Windows: {len(y)}"
            )

            print(
                f"Sequence shape: {seq.shape}"
            )

            print(
                f"HRV shape: {hrv.shape}"
            )

        if len(all_sequences) == 0:
            raise RuntimeError(
                "No windows were created from any subject."
            )

        X_seq = np.concatenate(
            all_sequences,
            axis=0
        )

        X_hrv = np.concatenate(
            all_hrv,
            axis=0
        )

        y = np.concatenate(
            all_labels,
            axis=0
        )

        groups = np.asarray(
            all_groups
        )

        return X_seq, X_hrv, y, groups


# ============================================================
# 9. TRAIN ONE LOSO FOLD
# ============================================================

def train_one_fold(
    X_train_seq,
    X_test_seq,
    X_train_hrv,
    X_test_hrv,
    y_train,
    y_test
):
    """Train and evaluate one LOSO fold."""

    # --------------------------------------------------------
    # Sequence scaler
    # Fit ONLY on training data
    # --------------------------------------------------------

    seq_scaler = StandardScaler()

    X_train_seq = seq_scaler.fit_transform(
        X_train_seq.reshape(
            -1,
            X_train_seq.shape[-1]
        )
    ).reshape(
        X_train_seq.shape
    )

    X_test_seq = seq_scaler.transform(
        X_test_seq.reshape(
            -1,
            X_test_seq.shape[-1]
        )
    ).reshape(
        X_test_seq.shape
    )

    # --------------------------------------------------------
    # HRV scaler
    # Fit ONLY on training data
    # --------------------------------------------------------

    hrv_scaler = StandardScaler()

    X_train_hrv = hrv_scaler.fit_transform(
        X_train_hrv
    )

    X_test_hrv = hrv_scaler.transform(
        X_test_hrv
    )

    # --------------------------------------------------------
    # Convert to tensors
    # --------------------------------------------------------

    X_train_seq = torch.tensor(
        X_train_seq,
        dtype=torch.float32
    )

    X_test_seq = torch.tensor(
        X_test_seq,
        dtype=torch.float32
    )

    X_train_hrv = torch.tensor(
        X_train_hrv,
        dtype=torch.float32
    )

    X_test_hrv = torch.tensor(
        X_test_hrv,
        dtype=torch.float32
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.long
    )

    y_test_tensor = torch.tensor(
        y_test,
        dtype=torch.long
    )

    # --------------------------------------------------------
    # DataLoader
    # --------------------------------------------------------

    train_dataset = TensorDataset(
        X_train_seq,
        X_train_hrv,
        y_train_tensor
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    # --------------------------------------------------------
    # Class weights
    # --------------------------------------------------------

    classes = np.unique(y_train)

    class_weights = np.ones(
        3,
        dtype=np.float32
    )

    calculated_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    class_weights[classes] = (
        calculated_weights
    )

    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float32,
        device=DEVICE
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = StressClassifier(
        seq_channels=2,
        hrv_dim=3,
        n_classes=3
    ).to(DEVICE)

    # --------------------------------------------------------
    # Loss + Optimizer
    # --------------------------------------------------------

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(1, EPOCHS + 1):

        model.train()

        running_loss = 0.0
        num_batches = 0

        for batch_seq, batch_hrv, batch_y in train_loader:

            batch_seq = batch_seq.to(DEVICE)
            batch_hrv = batch_hrv.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(
                batch_seq,
                batch_hrv
            )

            loss = criterion(
                outputs,
                batch_y
            )

            loss.backward()

            optimizer.step()

            running_loss += loss.item()
            num_batches += 1

        avg_loss = (
            running_loss / max(num_batches, 1)
        )

        print(
            f"Epoch {epoch:02d}/{EPOCHS} "
            f"| Loss: {avg_loss:.4f}"
        )

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    model.eval()

    with torch.no_grad():

        outputs = model(
            X_test_seq.to(DEVICE),
            X_test_hrv.to(DEVICE)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        ).cpu().numpy()

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    return accuracy, y_test, predictions


# ============================================================
# 10. LOSO VALIDATION
# ============================================================

def run_loso(
    X_seq,
    X_hrv,
    y,
    groups
):
    """Leave-One-Subject-Out validation."""

    logo = LeaveOneGroupOut()

    fold_accuracies = []

    fold_reports = []

    splits = logo.split(
        X_seq,
        y,
        groups
    )

    for fold, (
        train_idx,
        test_idx
    ) in enumerate(
        splits,
        start=1
    ):

        test_subject = groups[test_idx][0]

        print("\n" + "=" * 70)
        print(
            f"LOSO FOLD {fold}"
        )
        print(
            f"Test Subject: {test_subject}"
        )
        print("=" * 70)

        # ----------------------------------------------------
        # Split
        # ----------------------------------------------------

        X_train_seq = X_seq[train_idx]
        X_test_seq = X_seq[test_idx]

        X_train_hrv = X_hrv[train_idx]
        X_test_hrv = X_hrv[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        print(
            f"Train windows: {len(y_train)}"
        )

        print(
            f"Test windows: {len(y_test)}"
        )

        # ----------------------------------------------------
        # Train fold
        # ----------------------------------------------------

        accuracy, y_true, predictions = (
            train_one_fold(
                X_train_seq,
                X_test_seq,
                X_train_hrv,
                X_test_hrv,
                y_train,
                y_test
            )
        )

        fold_accuracies.append(
            accuracy
        )

        print(
            f"\nFold Accuracy: "
            f"{accuracy * 100:.2f}%"
        )

        # ----------------------------------------------------
        # Classification report
        # ----------------------------------------------------

        report = classification_report(
            y_true,
            predictions,
            labels=[0, 1, 2],
            target_names=[
                "Baseline",
                "Stress",
                "Amusement"
            ],
            zero_division=0
        )

        fold_reports.append(
            report
        )

        print("\nClassification Report:")
        print(report)

    # ========================================================
    # FINAL RESULTS
    # ========================================================

    fold_accuracies = np.asarray(
        fold_accuracies,
        dtype=np.float64
    )

    mean_accuracy = np.mean(
        fold_accuracies
    )

    std_accuracy = np.std(
        fold_accuracies
    )

    print("\n" + "=" * 70)
    print("FINAL LOSO RESULTS")
    print("=" * 70)

    for i, acc in enumerate(
        fold_accuracies,
        start=1
    ):
        print(
            f"Fold {i:02d}: "
            f"{acc * 100:.2f}%"
        )

    print("-" * 70)

    print(
        f"Mean Accuracy: "
        f"{mean_accuracy * 100:.2f}%"
    )

    print(
        f"Std Accuracy: "
        f"{std_accuracy * 100:.2f}%"
    )

    print("=" * 70)

    return fold_accuracies, fold_reports


# ============================================================
# 11. MAIN
# ============================================================

def main():

    print("\n" + "=" * 70)
    print("WESAD STRESS DETECTION PIPELINE")
    print("=" * 70)

    print(
        f"Device: {DEVICE}"
    )

    print(
        f"Target Sampling Rate: "
        f"{TARGET_SAMPLING_RATE} Hz"
    )

    print(
        f"Window: {WINDOW_SIZE} seconds"
    )

    print(
        f"Step: {STEP_SIZE} seconds"
    )

    print(
        "Input: EDA + BVP"
    )

    print(
        "HRV Features: Mean RR, SDNN, RMSSD"
    )

    print(
        "Classes: Baseline, Stress, Amusement"
    )

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    dataset_path = (
        download_wesad_dataset()
    )

    # --------------------------------------------------------
    # Pipeline
    # --------------------------------------------------------

    pipeline = WESADPipeline(
        dataset_path
    )

    # --------------------------------------------------------
    # Build windows
    # --------------------------------------------------------

    X_seq, X_hrv, y, groups = (
        pipeline.build_all_subject_windows()
    )

    # --------------------------------------------------------
    # Dataset summary
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("DATASET SUMMARY")
    print("=" * 70)

    print(
        f"X_seq shape : {X_seq.shape}"
    )

    print(
        f"X_hrv shape : {X_hrv.shape}"
    )

    print(
        f"y shape     : {y.shape}"
    )

    print(
        f"groups shape: {groups.shape}"
    )

    print(
        "\nClass distribution:"
    )

    class_names = [
        "Baseline",
        "Stress",
        "Amusement"
    ]

    for class_id, class_name in enumerate(
        class_names
    ):

        count = int(
            np.sum(y == class_id)
        )

        print(
            f"{class_name}: "
            f"{count}"
        )

    print(
        "\nSubjects found:"
    )

    print(
        np.unique(groups)
    )

    # --------------------------------------------------------
    # LOSO
    # --------------------------------------------------------

    results = run_loso(
        X_seq,
        X_hrv,
        y,
        groups
    )

    return results


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    fold_accuracies, fold_reports = main()



WESAD STRESS DETECTION PIPELINE
Device: cpu
Target Sampling Rate: 32 Hz
Window: 60 seconds
Step: 10 seconds
Input: EDA + BVP
HRV Features: Mean RR, SDNN, RMSSD
Classes: Baseline, Stress, Amusement
WESAD DATASET SETUP

WESAD ZIP already exists:
/content/WESAD.zip

WESAD appears to be already extracted.

S2.pkl found:
/content/WESAD/S2/S2.pkl

Dataset root:
/content/WESAD

PROCESSING S2

Loading S2
File: /content/WESAD/S2/S2.pkl
Windows: 230
Sequence shape: (230, 1920, 2)
HRV shape: (230, 3)

PROCESSING S3

Loading S3
File: /content/WESAD/S3/S3.pkl
Windows: 233
Sequence shape: (233, 1920, 2)
HRV shape: (233, 3)

PROCESSING S4

Loading S4
File: /content/WESAD/S4/S4.pkl
Windows: 234
Sequence shape: (234, 1920, 2)
HRV shape: (234, 3)

PROCESSING S5

Loading S5
File: /content/WESAD/S5/S5.pkl
Windows: 239
Sequence shape: (239, 1920, 2)
HRV shape: (239, 3)

PROCESSING S6

Loading S6
File: /content/WESAD/S6/S6.pkl
Windows: 238
Sequence shape: (238, 1920, 2)
HRV shape: (238, 3)

PROCESSING S7

